## USPTO ODP C# Notebook — Retrieving IP Data

Reference: [Getting Started — USPTO Open Data APIs](https://data.uspto.gov/apis/getting-started)


In [ ]:
// load nuget packages
#r "nuget: DotNetEnv, 2.5.*"
#r "nuget: System.Net.Http.Json, 6.0.0"

// load nuget packages for Azure Key Vault
#r "nuget: Azure.Identity, 1.5.0"
#r "nuget: Azure.Security.KeyVault.Secrets, 4.3.0"

// import namespaces
using System.Collections.Generic;
using System.Linq;
using System.Net.Http;
using System.Net.Http.Headers;
using DotNetEnv;
using System.Text.Json;
using System.Threading.Tasks;
using Azure.Identity;
using Azure.Security.KeyVault.Secrets;

In [ ]:
// get api key from Azure Key Vault. the secret name is "USPTO-ApiKey"
var kvVaultUri = "https://tdg-kv.vault.azure.net/";
var secretName = "USPTO-ApiKey";
var credential = new DefaultAzureCredential();

In [ ]:
// inspect identity being used
byte[] Base64UrlDecode(string input)
{
	var s = input.Replace('-', '+').Replace('_', '/');
	switch (s.Length % 4)
	{
		case 2: s += "=="; break;
		case 3: s += "="; break;
	}
	return System.Convert.FromBase64String(s);
}

var token = credential.GetToken(new Azure.Core.TokenRequestContext(new[] { "https://vault.azure.net/.default" })).Token;
var claims = System.Text.Json.JsonSerializer.Deserialize<Dictionary<string, object>>(Base64UrlDecode(token.Split('.')[1]));
Console.WriteLine($"appid: {claims.GetValueOrDefault("appid")}");
Console.WriteLine($"oid: {claims.GetValueOrDefault("oid")}");
Console.WriteLine($"upn: {claims.GetValueOrDefault("upn")}");
Console.WriteLine($"preferred_username: {claims.GetValueOrDefault("preferred_username")}");
Console.WriteLine($"tid: {claims.GetValueOrDefault("tid")}");

In [ ]:
var kvClient = new SecretClient(new Uri(kvVaultUri), credential);
KeyVaultSecret secret = kvClient.GetSecret(secretName);
string apiKey = secret.Value;
Console.WriteLine($"API Key from Azure Key Vault: {apiKey}");